In [3]:
import xml.etree.ElementTree as ET
from xml.dom import minidom
import numpy as np
import os

In [4]:
def prettify(element):
	"""Return a pretty-printed XML string for the Element."""
	rough_string = ET.tostring(element, 'utf-8')
	reparsed = minidom.parseString(rough_string)
	pretty_xml = reparsed.toprettyxml(indent="\t")
	pretty_xml = '\n'.join(line for line in pretty_xml.split('\n') if line.strip())
	return pretty_xml

In [5]:
def generate_vehicle_flows_from_triangular_FD(input_file, output_file):
	'''
	num_lanes: number of lanes
	max_time: simulation time in seconds
	desire_Q: desired flow rate in veh/hr
	dev: deviation of speed in percentage
	traffic_cond: free or congested
	input_file: input route file
	output_file: output vehicle file
	'''
	num_lanes = 6
	max_time = 600
	dev = 0.05
	traffic_cond = "free"
	insert_freq = 10 # every 1/10=0.1s
	desire_Q = 1.0 # veh/s per lane
	rho = 0.1
	avg_speed = 10 # m/s
	emit_prob = desire_Q  / insert_freq

	tree = ET.parse(input_file)
	root = tree.getroot()
	generated_vehicles = []
	route_id = 'e1_to_e6' # please specify the route id

	for ts in range(max_time*insert_freq):
		for lane in range(num_lanes):
			if np.random.rand() > emit_prob:
				continue
			vehicle_type = "PKW"
			vehicle = ET.Element("vehicle")
			vehicle.set("id", "veh_{}.{}".format(lane, ts))
			vehicle.set("type", vehicle_type)
			vehicle.set("route", route_id)
			vehicle.set("depart", str(ts/insert_freq))
			vehicle.set("departLane", str(lane))
			vehicle.set("departPos", "40")
			depart_speed = np.random.normal(avg_speed, avg_speed*dev)
			# add noise to depart speed
			if depart_speed < max(0., avg_speed - 2*avg_speed*dev):
				depart_speed = max(0., avg_speed - 2*avg_speed*dev)
			if depart_speed > avg_speed + 2*avg_speed*dev:
				depart_speed = avg_speed + 2*avg_speed*dev
			depart_speed = round(depart_speed, 1)
			vehicle.set("departSpeed", str(depart_speed))
			# you can set the arrival speed based on the traffic condition
			if traffic_cond == "congested":
				vehicle.set("arrivalSpeed", str(depart_speed))
			root.append(vehicle)
			generated_vehicles.append(vehicle)
	pretty_xml = prettify(root)
	with open(output_file, 'w') as f:
		f.write(pretty_xml)

In [6]:
# input file only specify the route and vType, like:
'''
<?xml version="1.0" encoding="UTF-8"?>

<additional xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:noNamespaceSchemaLocation="http://sumo.dlr.de/xsd/additional_file.xsd">
    <route id="e1_to_e6" edges="e1 e2 e3 e4 e5 e6"/>
    <route id="er1_to_e6" edges="er1 e3 e4 e5 e6"/>
    <vType id="PKW" vClass="passenger"/>
    <vType id="LKW" vClass="truck" carFollowModel="IDM" tau="1.6" minGap="2.0" accel="0.73" decel="1.67" desiredMaxSpeed="24.59" stepping="0.1"/>
</additional>
'''
generate_vehicle_flows_from_triangular_FD("v2_vehicles_meta.rou.xml", "v2_vehicles_test.rou.xml")